In [3]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [8]:
B = 4
D = 3

image_features = torch.tensor([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
    [1.0, 1.0, 0.0],
])

text_features = torch.tensor([
    [0.9, 0.1, 0.0],
    [0.1, 0.9, 0.0],
    [0.0, 0.1, 0.9],
    [0.8, 0.8, 0.1],
])

image_features @ text_features.T should be 4,4, 
mat mul the 4 images with the 4 texts, each having 3 features.
Multiply each feature for pairwise number representing
the mat mul'd image and text. 2,1 should represent the matrix multiplied
version of the features of the 3rd image with the features of the 
2nd text example. It normalizes so that it can just filter for 
semantic directional alignment rather than magnitude. 

In [14]:
image_features_norm = torch.nn.functional.normalize(image_features)
text_features_norm = torch.nn.functional.normalize(text_features)

print(image_features_norm)
print(text_features_norm)

tensor([[1.0000, 0.0000, 0.0000],
        [0.0000, 1.0000, 0.0000],
        [0.0000, 0.0000, 1.0000],
        [0.7071, 0.7071, 0.0000]])
tensor([[0.9939, 0.1104, 0.0000],
        [0.1104, 0.9939, 0.0000],
        [0.0000, 0.1104, 0.9939],
        [0.7044, 0.7044, 0.0880]])


In [17]:
similarity_it = image_features_norm @ text_features_norm.T
similarity_ti = similarity_it.T

print(similarity_it)
print(similarity_ti)

tensor([[0.9939, 0.1104, 0.0000, 0.7044],
        [0.1104, 0.9939, 0.1104, 0.7044],
        [0.0000, 0.0000, 0.9939, 0.0880],
        [0.7809, 0.7809, 0.0781, 0.9961]])
tensor([[0.9939, 0.1104, 0.0000, 0.7809],
        [0.1104, 0.9939, 0.0000, 0.7809],
        [0.0000, 0.1104, 0.9939, 0.0781],
        [0.7044, 0.7044, 0.0880, 0.9961]])


Softmax softens the distribution too much, we need to add temperature to control sharpness.

In [19]:
targets = torch.arange(B)

F.cross_entropy(similarity_it, targets)
probs = similarity_it.softmax(dim=1)

print(similarity_it[0])
print(probs[0])

tensor([0.9939, 0.1104, 0.0000, 0.7044])
tensor([0.3949, 0.1632, 0.1462, 0.2957])


In [23]:
logit_scale = torch.tensor(2.6592)  # roughly log(1 / 0.07)

scaled_similarity = logit_scale.exp() * similarity_it

print("scale:", logit_scale.exp())
print("raw probs:", similarity_it[0].softmax(dim=0))
print("scaled probs:", scaled_similarity[0].softmax(dim=0))

scale: tensor(14.2849)
raw probs: tensor([0.3949, 0.1632, 0.1462, 0.2957])
scaled probs: tensor([9.8426e-01, 3.2533e-06, 6.7177e-07, 1.5738e-02])


In [31]:
def clip_loss(
    image_features: torch.Tensor,
    text_features: torch.Tensor,
    logit_scale: torch.Tensor,
) -> torch.Tensor:

    # 1. normalize image features
    image_features_norm = torch.nn.functional.normalize(image_features)
    
    # 2. normalize text features
    text_features_norm = torch.nn.functional.normalize(text_features)

    # 3. construct B x B similarity matrix
    similarity_it = image_features_norm @ text_features_norm.T
    similarity_ti = similarity_it.T

    # 4. construct targets
    batch_size = image_features.shape[0]
    targets = torch.arange(batch_size, device=image_features.device)

    # 5. image -> text cross entropy
    scale = logit_scale.exp()
    logits_it = scale * similarity_it
    ce_it = F.cross_entropy(logits_it, targets)
    
    # 6. text -> image cross entropy
    logits_ti = scale * similarity_ti
    ce_ti = F.cross_entropy(logits_ti, targets)

    # 7. symmetric average
    loss = (ce_it + ce_ti) / 2

    return loss

In [32]:
loss = clip_loss(
    image_features,
    text_features,
    torch.tensor(2.6592),
)

print(loss)

tensor(0.0305)


In [33]:
shuffled_text_features = text_features[[2, 0, 3, 1]]

shuffled_loss = clip_loss(
    image_features,
    shuffled_text_features,
    torch.tensor(2.6592),
)

print("aligned loss:", loss.item())
print("shuffled loss:", shuffled_loss.item())

aligned loss: 0.030478760600090027
shuffled loss: 10.738485336303711


Testing a toy dataset to isolate model ability to distinguish visual features to semantic meaning. Lets start with shapes. Enough combinations to have a sufficiently large dataset while retaining distinct caption labels, with distinct visual features matching its corresponding semantic representation

In [86]:
colors = ["red", "blue", "green"]
shapes = ["circle", "triangle", "square"]
sizes = ["small", "large"]
positions = ["left", "center", "right"]

held_out_pairs = {
    ("red", "triangle"),
    ("blue", "square"),
}

test_combinations = []
train_combinations = []

for color in colors:
    for shape in shapes:
        for size in sizes:
            for position in positions:
                if (color, shape) in held_out_pairs:
                    test_combinations.append((color, shape, size, position))
                else:
                    train_combinations.append((color, shape, size, position))
                

def make_caption(combo):
    color, shape, size, position = combo

    if position == "center":
        location = "in the center"
    else:
        location = f"on the {position}"

    return f"a {size} {color} {shape} {location}"

In [52]:
print(f"train: {len(train_combinations)}")
print(f"test: {len(test_combinations)}")

train: 42
test: 12


In [87]:
test = make_caption(test_combinations[1])
test

'a small red triangle in the center'

In [55]:
assert all(
    (color, shape) not in held_out_pairs
    for color, shape, size, position in train_combinations
)

assert all(
    (color, shape) in held_out_pairs
    for color, shape, size, position in test_combinations
)

Rendering

Red: (r, 0, 0) where r ∈ [160, 255]
Green: (0, g, 0) where g ∈ [160, 255]
Blue: (0, 0, b) where b ∈ [160, 255]

Bounding-box width size (disjoint)
small: 8-14 pixels in width
large: 22-28 pixels in width

Vertical position remains independent for now, set y = 32

Horizontal regions:
left: x ∈ [10, 18]
center: x ∈ [26, 38]
right: x ∈ [46, 54]

Clip/constrain the region depending on shape size.

Start with a black background (0, 0, 0) for all non-shape pixels.

Keep the ranges disjoint to prevent label ambiguity.

In [88]:
import random
import math
from PIL import Image, ImageDraw


colors = ["red", "blue", "green"]
shapes = ["circle", "triangle", "square"]
sizes = ["small", "large"]
positions = ["left", "center", "right"]

held_out_pairs = {
    ("red", "triangle"),
    ("blue", "square"),
}


def make_caption(combo):
    color, shape, size, position = combo

    if position == "center":
        location = "in the center"
    else:
        location = f"on the {position}"

    return f"a {size} {color} {shape} {location}"


def build_splits():
    train_combinations = []
    test_combinations = []

    for color in colors:
        for shape in shapes:
            for size in sizes:
                for position in positions:
                    combo = (color, shape, size, position)
                    if (color, shape) in held_out_pairs:
                        test_combinations.append(combo)
                    else:
                        train_combinations.append(combo)

    assert len(train_combinations) == 42
    assert len(test_combinations) == 12
    assert len(train_combinations) + len(test_combinations) == 54

    assert all(
        (color, shape) not in held_out_pairs
        for color, shape, size, position in train_combinations
    )
    assert all(
        (color, shape) in held_out_pairs
        for color, shape, size, position in test_combinations
    )

    return train_combinations, test_combinations


def sample_color(color_name):
    if color_name == "red":
        return (random.randint(160, 255), 0, 0)
    elif color_name == "green":
        return (0, random.randint(160, 255), 0)
    elif color_name == "blue":
        return (0, 0, random.randint(160, 255))
    else:
        raise ValueError(f"Unknown color: {color_name}")


def sample_size(size_name):
    if size_name == "small":
        return random.randint(8, 14)
    elif size_name == "large":
        return random.randint(22, 28)
    else:
        raise ValueError(f"Unknown size: {size_name}")


def position_range(position_name):
    if position_name == "left":
        return (10, 18)
    elif position_name == "center":
        return (26, 38)
    elif position_name == "right":
        return (46, 54)
    else:
        raise ValueError(f"Unknown position: {position_name}")


def sample_x(position_name, size, image_size=64):
    # Keep shape inside image.
    half = math.ceil(size / 2)
    valid_min = half
    valid_max = image_size - half

    region_min, region_max = position_range(position_name)

    lo = max(region_min, valid_min)
    hi = min(region_max, valid_max)

    assert lo <= hi, (
        f"Empty x-intersection for position={position_name}, "
        f"size={size}, region=({region_min},{region_max}), "
        f"valid=({valid_min},{valid_max})"
    )

    return random.randint(lo, hi)


def draw_circle(draw, x, y, size, fill):
    half = size / 2
    bbox = [x - half, y - half, x + half, y + half]
    draw.ellipse(bbox, fill=fill)


def draw_square(draw, x, y, size, fill):
    half = size / 2
    bbox = [x - half, y - half, x + half, y + half]
    draw.rectangle(bbox, fill=fill)


def draw_triangle(draw, x, y, size, fill):
    # Upright isosceles triangle inside a size x size box.
    half = size / 2

    top = (x, y - half)
    bottom_left = (x - half, y + half)
    bottom_right = (x + half, y + half)

    draw.polygon([top, bottom_left, bottom_right], fill=fill)


def render_shape(combo, image_size=64, return_metadata=False):
    color_name, shape_name, size_name, position_name = combo

    fill = sample_color(color_name)
    size = sample_size(size_name)
    x = sample_x(position_name, size, image_size=image_size)
    y = 32

    image = Image.new("RGB", (image_size, image_size), (0, 0, 0))
    draw = ImageDraw.Draw(image)

    if shape_name == "circle":
        draw_circle(draw, x, y, size, fill)
    elif shape_name == "square":
        draw_square(draw, x, y, size, fill)
    elif shape_name == "triangle":
        draw_triangle(draw, x, y, size, fill)
    else:
        raise ValueError(f"Unknown shape: {shape_name}")

    metadata = {
        "combo": combo,
        "caption": make_caption(combo),
        "rgb": fill,
        "size": size,
        "x": x,
        "y": y,
    }

    if return_metadata:
        return image, metadata
    return image


def check_all_position_size_intersections(image_size=64):
    failures = []

    for size_name in sizes:
        for position_name in positions:
            if size_name == "small":
                candidate_sizes = range(8, 15)
            else:
                candidate_sizes = range(22, 29)

            for s in candidate_sizes:
                half = math.ceil(s / 2)
                valid_min = half
                valid_max = image_size - half
                region_min, region_max = position_range(position_name)

                lo = max(region_min, valid_min)
                hi = min(region_max, valid_max)

                if lo > hi:
                    failures.append((size_name, position_name, s, (lo, hi)))

    return failures


def save_example_samples(out_dir="samples"):
    import os
    os.makedirs(out_dir, exist_ok=True)

    example_combos = [
        ("red", "square", "small", "left"),
        ("blue", "circle", "large", "center"),
        ("green", "triangle", "small", "right"),
    ]

    all_metadata = []

    for i, combo in enumerate(example_combos):
        img, meta = render_shape(combo, return_metadata=True)
        path = os.path.join(out_dir, f"sample_{i}_{'_'.join(combo)}.png")
        img.save(path)
        meta["path"] = path
        all_metadata.append(meta)

    return all_metadata

In [77]:
train_combinations, test_combinations = build_splits()

print("Train:", len(train_combinations))
print("Test:", len(test_combinations))
print("Example train captions:")
for combo in train_combinations[:3]:
    print(make_caption(combo))

print("\nAll test captions:")
for combo in test_combinations:
    print(make_caption(combo))

failures = check_all_position_size_intersections()
print("\nIntersection failures:", failures)

metadata = save_example_samples()
for m in metadata:
    print(m)

Train: 42
Test: 12
Example train captions:
a small red circle on the left
a small red circle in the center
a small red circle on the right

All test captions:
a small red triangle on the left
a small red triangle in the center
a small red triangle on the right
a large red triangle on the left
a large red triangle in the center
a large red triangle on the right
a small blue square on the left
a small blue square in the center
a small blue square on the right
a large blue square on the left
a large blue square in the center
a large blue square on the right

Intersection failures: []
{'combo': ('red', 'square', 'small', 'left'), 'caption': 'a small red square on the left', 'rgb': (246, 0, 0), 'size': 10, 'x': 14, 'y': 32, 'path': 'samples/sample_0_red_square_small_left.png'}
{'combo': ('blue', 'circle', 'large', 'center'), 'caption': 'a large blue circle in the center', 'rgb': (0, 0, 174), 'size': 24, 'x': 30, 'y': 32, 'path': 'samples/sample_1_blue_circle_large_center.png'}
{'combo': ('g

Hypothesis:

A tiny CLIP-style model trained on the 42 observed combinations will learn enough shared attribute structure to retrieve held-out blue-circle and green-square compositions above one-attribute distractors.

In [78]:
import math
import random
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import pil_to_tensor


# ------------------------------------------------------------
# 1. DATASET
# ------------------------------------------------------------

class ShapeDataset(Dataset):
    def __init__(self, combinations):
        self.combinations = combinations

    def __len__(self):
        return len(self.combinations)

    def __getitem__(self, idx):
        combo = self.combinations[idx]

        # New rendering every time this sample is requested
        image = render_shape(combo)

        # PIL -> tensor, normalize pixels to [0, 1]
        image = pil_to_tensor(image).float() / 255.0

        caption = make_caption(combo)

        return image, caption, combo


# ------------------------------------------------------------
# 2. TOKENIZER
# ------------------------------------------------------------

class SimpleTokenizer:
    def __init__(self, captions):
        words = set()

        for caption in captions:
            words.update(caption.lower().split())

        # padding token is index 0
        self.word_to_id = {"<pad>": 0}

        for word in sorted(words):
            self.word_to_id[word] = len(self.word_to_id)

        self.id_to_word = {
            idx: word
            for word, idx in self.word_to_id.items()
        }

    @property
    def vocab_size(self):
        return len(self.word_to_id)

    def encode(self, caption):
        return [
            self.word_to_id[word]
            for word in caption.lower().split()
        ]

    def batch_encode(self, captions):
        encoded = [self.encode(c) for c in captions]

        max_len = max(len(x) for x in encoded)

        tokens = torch.zeros(
            len(encoded),
            max_len,
            dtype=torch.long
        )

        mask = torch.zeros(
            len(encoded),
            max_len,
            dtype=torch.float32
        )

        for i, sequence in enumerate(encoded):
            length = len(sequence)

            tokens[i, :length] = torch.tensor(sequence)
            mask[i, :length] = 1.0

        return tokens, mask


# Build tokenizer from ALL possible vocabulary-containing captions.
#
# This is okay because we are not learning semantic pairings from the
# held-out examples here. We only need held-out words like "blue" and
# "circle" to exist in the vocabulary.
all_combinations = train_combinations + test_combinations

all_captions = [
    make_caption(combo)
    for combo in all_combinations
]

tokenizer = SimpleTokenizer(all_captions)


# ------------------------------------------------------------
# 3. IMAGE ENCODER
# ------------------------------------------------------------

class ImageEncoder(nn.Module):
    def __init__(self, embed_dim=64):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(2),       # 64 -> 32

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(2),       # 32 -> 16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(2),       # 16 -> 8
        )

        # Preserve spatial layout: DO NOT global-average-pool.
        self.projection = nn.Linear(
            64 * 8 * 8,
            embed_dim
        )

    def forward(self, images):
        x = self.features(images)

        # [B, 64, 8, 8] -> [B, 4096]
        x = x.flatten(start_dim=1)

        x = self.projection(x)

        return x


# ------------------------------------------------------------
# 4. TEXT ENCODER
# ------------------------------------------------------------

class TextEncoder(nn.Module):
    def __init__(self, vocab_size, token_dim=32, embed_dim=64):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            token_dim,
            padding_idx=0
        )

        self.projection = nn.Linear(
            token_dim,
            embed_dim
        )

    def forward(self, tokens, mask):

        # [B, L] -> [B, L, token_dim]
        x = self.embedding(tokens)

        # Mask padding positions
        mask = mask.unsqueeze(-1)    # [B, L, 1]

        x = x * mask

        summed = x.sum(dim=1)

        lengths = mask.sum(dim=1).clamp(min=1)

        # Bag-of-token / mean-pooled text representation
        pooled = summed / lengths

        return self.projection(pooled)


# ------------------------------------------------------------
# 5. TINY CLIP
# ------------------------------------------------------------

class TinyCLIP(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim=64,
        token_dim=32,
        initial_temperature=0.07,
    ):
        super().__init__()

        self.image_encoder = ImageEncoder(
            embed_dim=embed_dim
        )

        self.text_encoder = TextEncoder(
            vocab_size=vocab_size,
            token_dim=token_dim,
            embed_dim=embed_dim
        )

        # CLIP usually learns logit scale rather than tau directly.
        self.logit_scale = nn.Parameter(
            torch.tensor(
                math.log(1.0 / initial_temperature)
            )
        )

    def encode_image(self, images):
        embeddings = self.image_encoder(images)

        return F.normalize(
            embeddings,
            dim=-1
        )

    def encode_text(self, tokens, mask):
        embeddings = self.text_encoder(tokens, mask)

        return F.normalize(
            embeddings,
            dim=-1
        )

    def forward(self, images, tokens, mask):

        image_embeddings = self.encode_image(images)
        text_embeddings = self.encode_text(tokens, mask)

        scale = self.logit_scale.exp()

        logits = scale * (
            image_embeddings @ text_embeddings.T
        )

        return logits


# ------------------------------------------------------------
# 6. CLIP LOSS
# ------------------------------------------------------------

def clip_loss(logits):

    batch_size = logits.shape[0]

    targets = torch.arange(
        batch_size,
        device=logits.device
    )

    image_to_text = F.cross_entropy(
        logits,
        targets
    )

    text_to_image = F.cross_entropy(
        logits.T,
        targets
    )

    return (image_to_text + text_to_image) / 2


# ------------------------------------------------------------
# 7. COLLATE
# ------------------------------------------------------------

def collate_fn(batch):

    images, captions, combos = zip(*batch)

    images = torch.stack(images)

    tokens, mask = tokenizer.batch_encode(captions)

    return (
        images,
        tokens,
        mask,
        list(captions),
        list(combos),
    )


# ------------------------------------------------------------
# 8. DATA LOADER
# ------------------------------------------------------------

train_dataset = ShapeDataset(
    train_combinations
)

train_loader = DataLoader(
    train_dataset,
    batch_size=len(train_dataset),   # 42
    shuffle=True,
    collate_fn=collate_fn,
)


# ------------------------------------------------------------
# 9. MODEL + OPTIMIZER
# ------------------------------------------------------------

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

model = TinyCLIP(
    vocab_size=tokenizer.vocab_size,
    embed_dim=64,
    token_dim=32
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)


# ------------------------------------------------------------
# 10. TRAIN RETRIEVAL ACCURACY
# ------------------------------------------------------------

def retrieval_accuracy(logits):

    targets = torch.arange(
        logits.shape[0],
        device=logits.device
    )

    image_predictions = logits.argmax(dim=1)

    text_predictions = logits.argmax(dim=0)

    image_to_text = (
        image_predictions == targets
    ).float().mean().item()

    text_to_image = (
        text_predictions == targets
    ).float().mean().item()

    return image_to_text, text_to_image


# ------------------------------------------------------------
# 11. TRAINING LOOP
# ------------------------------------------------------------

num_epochs = 500

history = {
    "loss": [],
    "i2t": [],
    "t2i": [],
    "temperature": []
}

for epoch in range(num_epochs):

    model.train()

    for (
        images,
        tokens,
        mask,
        captions,
        combos
    ) in train_loader:

        images = images.to(device)
        tokens = tokens.to(device)
        mask = mask.to(device)

        optimizer.zero_grad()

        logits = model(
            images,
            tokens,
            mask
        )

        loss = clip_loss(logits)

        loss.backward()

        optimizer.step()

        i2t, t2i = retrieval_accuracy(logits)

        temperature = (
            1.0 / model.logit_scale.exp().item()
        )

        history["loss"].append(loss.item())
        history["i2t"].append(i2t)
        history["t2i"].append(t2i)
        history["temperature"].append(temperature)

    if (
        epoch == 0
        or (epoch + 1) % 25 == 0
    ):
        print(
            f"Epoch {epoch + 1:4d} | "
            f"Loss {loss.item():.4f} | "
            f"I2T {i2t:.3f} | "
            f"T2I {t2i:.3f} | "
            f"tau {temperature:.4f}"
        )

Epoch    1 | Loss 3.9395 | I2T 0.048 | T2I 0.071 | tau 0.0700
Epoch   25 | Loss 0.9843 | I2T 0.762 | T2I 0.762 | tau 0.0699
Epoch   50 | Loss 0.5028 | I2T 0.952 | T2I 0.929 | tau 0.0694
Epoch   75 | Loss 0.2951 | I2T 0.976 | T2I 1.000 | tau 0.0689
Epoch  100 | Loss 0.1307 | I2T 1.000 | T2I 1.000 | tau 0.0685
Epoch  125 | Loss 0.0870 | I2T 1.000 | T2I 1.000 | tau 0.0682
Epoch  150 | Loss 0.0599 | I2T 1.000 | T2I 1.000 | tau 0.0679
Epoch  175 | Loss 0.0506 | I2T 1.000 | T2I 1.000 | tau 0.0676
Epoch  200 | Loss 0.0400 | I2T 1.000 | T2I 1.000 | tau 0.0674
Epoch  225 | Loss 0.0376 | I2T 1.000 | T2I 1.000 | tau 0.0672
Epoch  250 | Loss 0.0319 | I2T 1.000 | T2I 1.000 | tau 0.0670
Epoch  275 | Loss 0.0292 | I2T 1.000 | T2I 1.000 | tau 0.0668
Epoch  300 | Loss 0.0258 | I2T 1.000 | T2I 1.000 | tau 0.0666
Epoch  325 | Loss 0.0245 | I2T 1.000 | T2I 1.000 | tau 0.0664
Epoch  350 | Loss 0.0225 | I2T 1.000 | T2I 1.000 | tau 0.0662
Epoch  375 | Loss 0.0205 | I2T 1.000 | T2I 1.000 | tau 0.0661
Epoch  4

In [79]:
@torch.no_grad()
def encode_caption_bank(model, combinations, tokenizer, device):

    captions = [
        make_caption(combo)
        for combo in combinations
    ]

    tokens, mask = tokenizer.batch_encode(captions)

    tokens = tokens.to(device)
    mask = mask.to(device)

    text_embeddings = model.encode_text(
        tokens,
        mask
    )

    return captions, text_embeddings


@torch.no_grad()
def evaluate_held_out(
    model,
    test_combinations,
    all_combinations,
    tokenizer,
    device,
    samples_per_combo=20
):

    model.eval()

    captions, text_embeddings = encode_caption_bank(
        model,
        all_combinations,
        tokenizer,
        device
    )

    caption_to_index = {
        caption: i
        for i, caption in enumerate(captions)
    }

    correct = 0
    total = 0

    color_margins = []
    shape_margins = []

    results = []

    for combo in test_combinations:

        color, shape, size, position = combo

        correct_caption = make_caption(combo)

        for _ in range(samples_per_combo):

            image = render_shape(combo)

            image = (
                pil_to_tensor(image)
                .float()
                .div(255.0)
                .unsqueeze(0)
                .to(device)
            )

            image_embedding = model.encode_image(
                image
            )

            similarities = (
                image_embedding
                @ text_embeddings.T
            ).squeeze(0)

            pred_idx = similarities.argmax().item()

            predicted_caption = captions[pred_idx]

            if predicted_caption == correct_caption:
                correct += 1

            total += 1

            # -----------------------------------------
            # Attribute-controlled distractors
            # -----------------------------------------

            # Same shape/size/position, wrong color
            wrong_colors = [
                c
                for c in colors
                if c != color
            ]

            color_distractor_scores = []

            for wrong_color in wrong_colors:

                distractor = (
                    wrong_color,
                    shape,
                    size,
                    position
                )

                distractor_caption = make_caption(
                    distractor
                )

                idx = caption_to_index[
                    distractor_caption
                ]

                color_distractor_scores.append(
                    similarities[idx].item()
                )

            # Same color/size/position, wrong shape
            wrong_shapes = [
                s
                for s in shapes
                if s != shape
            ]

            shape_distractor_scores = []

            for wrong_shape in wrong_shapes:

                distractor = (
                    color,
                    wrong_shape,
                    size,
                    position
                )

                distractor_caption = make_caption(
                    distractor
                )

                idx = caption_to_index[
                    distractor_caption
                ]

                shape_distractor_scores.append(
                    similarities[idx].item()
                )

            correct_idx = caption_to_index[
                correct_caption
            ]

            correct_score = (
                similarities[correct_idx].item()
            )

            color_margin = (
                correct_score
                - max(color_distractor_scores)
            )

            shape_margin = (
                correct_score
                - max(shape_distractor_scores)
            )

            color_margins.append(color_margin)
            shape_margins.append(shape_margin)

            results.append({
                "combo": combo,
                "correct_caption": correct_caption,
                "predicted_caption": predicted_caption,
                "correct_score": correct_score,
                "color_margin": color_margin,
                "shape_margin": shape_margin,
            })

    accuracy = correct / total

    mean_color_margin = (
        sum(color_margins)
        / len(color_margins)
    )

    mean_shape_margin = (
        sum(shape_margins)
        / len(shape_margins)
    )

    return {
        "accuracy": accuracy,
        "color_margin": mean_color_margin,
        "shape_margin": mean_shape_margin,
        "results": results,
    }

In [80]:
evaluation = evaluate_held_out(
    model=model,
    test_combinations=test_combinations,
    all_combinations=all_combinations,
    tokenizer=tokenizer,
    device=device,
    samples_per_combo=20
)

print(
    "Held-out accuracy:",
    evaluation["accuracy"]
)

print(
    "Mean color margin:",
    evaluation["color_margin"]
)

print(
    "Mean shape margin:",
    evaluation["shape_margin"]
)

Held-out accuracy: 0.5
Mean color margin: 0.3211716516253849
Mean shape margin: -0.03515626080334187


Very interesting. The model seems to have learned color representations better than shape. A plausible explanation could be that the model might have overtrained on shape-color pairings in the training set, so for the blue circle held out label for example, I would expect blue square/blue triangle to be the most common errors because during training the model could have memorized those pairings. For green-square images it would be green-circle/green-triangle.

In [81]:
seen_eval = evaluate_held_out(
    model=model,
    test_combinations=train_combinations,
    all_combinations=all_combinations,
    tokenizer=tokenizer,
    device=device,
    samples_per_combo=20
)

print("Fresh seen accuracy:", seen_eval["accuracy"])
print("Fresh seen color margin:", seen_eval["color_margin"])
print("Fresh seen shape margin:", seen_eval["shape_margin"])

Fresh seen accuracy: 1.0
Fresh seen color margin: 0.3443158854331289
Fresh seen shape margin: 0.3156680511221999


So the model can recognize shape perfectly well on fresh stochastic renderings of seen combinations. That strongly weakens the hypothesis that the image encoder simply failed to learn shape.

In [82]:
from collections import Counter

pred_counts = Counter(
    r["predicted_caption"]
    for r in evaluation["results"]
)

for caption, count in pred_counts.most_common(15):
    print(count, caption)

20 a small red triangle on the left
20 a small red triangle in the center
20 a small red triangle on the right
20 a large red triangle on the left
20 a large red triangle in the center
20 a large red triangle on the right
20 a small blue circle on the left
20 a small blue circle in the center
20 a small blue circle on the right
20 a large blue circle on the left
20 a large blue circle in the center
20 a large blue circle on the right


For blue circles, the model almost always predicts the corresponding blue square with the same size and position, and green squares map essentially perfectly to green circles, again preserving size and position.

The model appears to have generalized at least three factors quite well:color, size, position

But it failed to recombine shape with color when that exact color–shape relation was excluded from training.

Hypothesis:

The learned representation factorizes some semantic attributes, but color and shape remain entangled in a way that prevents systematic recombination of unseen color–shape pairs.

However, we predicted blue-circle might become:

blue square OR blue triangle

But it overwhelmingly becomes blue square.

Similarly:

green square becomes green circle almost deterministically.

Why square rather than triangle for blue? Why circle rather than triangle for green?

We don't yet know. Possible explanations include learned joint color–shape associations, representation geometry, biases from optimization/initialization, or some interaction with the visual features.

Low variance + consistently poor held-out generalization: evidence that the failure is a systematic property of this setup—likely something about the data, objective, or architectural inductive biases consistently encourages seen color–shape associations over unseen recombination.

High variance + similar training performance: evidence of underspecification: the training objective does not determine how color and shape should be organized outside the observed combinations.

In [83]:
def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_one_seed(seed, num_epochs=500):
    set_seed(seed)

    model = TinyCLIP(
        vocab_size=tokenizer.vocab_size,
        embed_dim=64,
        token_dim=32
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=3e-4,
        weight_decay=1e-4
    )

    final_loss = None

    for epoch in range(num_epochs):
        model.train()

        for images, tokens, mask, captions, combos in train_loader:
            images = images.to(device)
            tokens = tokens.to(device)
            mask = mask.to(device)

            optimizer.zero_grad()

            logits = model(images, tokens, mask)
            loss = clip_loss(logits)

            loss.backward()
            optimizer.step()

            final_loss = loss.item()

    return model, final_loss

In [84]:
seeds = [0, 1, 2, 3, 4]
rows = []

for seed in seeds:

    model, train_loss = train_one_seed(seed)

    # Identical held-out renderings for every trained model
    set_seed(12345)
    held = evaluate_held_out(
        model,
        test_combinations,
        all_combinations,
        tokenizer,
        device,
        samples_per_combo=20
    )

    # Identical fresh-seen renderings for every model
    set_seed(54321)
    seen = evaluate_held_out(
        model,
        train_combinations,
        all_combinations,
        tokenizer,
        device,
        samples_per_combo=20
    )

    rows.append({
        "seed": seed,
        "train_loss": train_loss,
        "seen_accuracy": seen["accuracy"],
        "held_accuracy": held["accuracy"],
        "held_color_margin": held["color_margin"],
        "held_shape_margin": held["shape_margin"],
    })

    print(rows[-1])

{'seed': 0, 'train_loss': 0.012769918888807297, 'seen_accuracy': 1.0, 'held_accuracy': 0.7166666666666667, 'held_color_margin': 0.311303453023235, 'held_shape_margin': 0.11939506481091182}
{'seed': 1, 'train_loss': 0.013032598420977592, 'seen_accuracy': 1.0, 'held_accuracy': 0.48333333333333334, 'held_color_margin': 0.25535884288450084, 'held_shape_margin': 0.008966432760159175}
{'seed': 2, 'train_loss': 0.014786260202527046, 'seen_accuracy': 1.0, 'held_accuracy': 0.5, 'held_color_margin': 0.37058645431728415, 'held_shape_margin': -0.0031449427207310993}
{'seed': 3, 'train_loss': 0.014437171630561352, 'seen_accuracy': 1.0, 'held_accuracy': 0.5, 'held_color_margin': 0.30538624711334705, 'held_shape_margin': -0.03883597540358703}
{'seed': 4, 'train_loss': 0.013127416372299194, 'seen_accuracy': 1.0, 'held_accuracy': 0.5125, 'held_color_margin': 0.30333749850591024, 'held_shape_margin': 0.0754471609989802}


In [85]:
import pandas as pd

df = pd.DataFrame(rows)

print(df)
print("\nMean:")
print(df.mean(numeric_only=True))

print("\nStd:")
print(df.std(numeric_only=True))

   seed  train_loss  seen_accuracy  held_accuracy  held_color_margin  \
0     0    0.012770            1.0       0.716667           0.311303   
1     1    0.013033            1.0       0.483333           0.255359   
2     2    0.014786            1.0       0.500000           0.370586   
3     3    0.014437            1.0       0.500000           0.305386   
4     4    0.013127            1.0       0.512500           0.303337   

   held_shape_margin  
0           0.119395  
1           0.008966  
2          -0.003145  
3          -0.038836  
4           0.075447  

Mean:
seed                 2.000000
train_loss           0.013631
seen_accuracy        1.000000
held_accuracy        0.542500
held_color_margin    0.309194
held_shape_margin    0.032366
dtype: float64

Std:
seed                 1.581139
train_loss           0.000913
seen_accuracy        0.000000
held_accuracy        0.097912
held_color_margin    0.040989
held_shape_margin    0.063852
dtype: float64


After testing with a different held out example:

held_out_pairs = {
    ("red", "triangle"),
    ("blue", "square"),
}

Our original hypothesis for the model overfitting on color-shape combinations in the training set is weakened.

Compositional generalization in this toy CLIP system is highly dependent on the particular training co-occurrence structure; some unseen color–shape combinations are recoverable while others are systematically mapped toward seen combinations.

In [89]:
def summarize_pair(eval_result, pair):
    subset = [
        r for r in eval_result["results"]
        if r["combo"][:2] == pair
    ]

    accuracy = sum(
        r["predicted_caption"] == r["correct_caption"]
        for r in subset
    ) / len(subset)

    color_margin = sum(
        r["color_margin"]
        for r in subset
    ) / len(subset)

    shape_margin = sum(
        r["shape_margin"]
        for r in subset
    ) / len(subset)

    return {
        "accuracy": accuracy,
        "color_margin": color_margin,
        "shape_margin": shape_margin,
    }

In [90]:
import pandas as pd

seeds = [0, 1, 2, 3, 4]

held_out_pairs = {
    ("red", "triangle"),
    ("blue", "square"),
}

rows = []

for seed in seeds:

    model, train_loss = train_one_seed(seed)

    # Fixed evaluation renderings across seeds
    set_seed(12345)

    held = evaluate_held_out(
        model=model,
        test_combinations=test_combinations,
        all_combinations=all_combinations,
        tokenizer=tokenizer,
        device=device,
        samples_per_combo=20,
    )

    red_triangle = summarize_pair(
        held,
        ("red", "triangle")
    )

    blue_square = summarize_pair(
        held,
        ("blue", "square")
    )

    rows.append({
        "seed": seed,

        "red_triangle_acc":
            red_triangle["accuracy"],

        "red_triangle_color_margin":
            red_triangle["color_margin"],

        "red_triangle_shape_margin":
            red_triangle["shape_margin"],

        "blue_square_acc":
            blue_square["accuracy"],

        "blue_square_color_margin":
            blue_square["color_margin"],

        "blue_square_shape_margin":
            blue_square["shape_margin"],
    })


df_pairs = pd.DataFrame(rows)

print(df_pairs)

print("\nMean:")
print(df_pairs.mean(numeric_only=True))

print("\nStd:")
print(df_pairs.std(numeric_only=True))

   seed  red_triangle_acc  red_triangle_color_margin  \
0     0          1.000000                   0.256650   
1     1          0.966667                   0.194008   
2     2          1.000000                   0.299739   
3     3          1.000000                   0.243220   
4     4          1.000000                   0.249970   

   red_triangle_shape_margin  blue_square_acc  blue_square_color_margin  \
0                   0.245484         0.433333                  0.365993   
1                   0.179914         0.000000                  0.315578   
2                   0.123426         0.000000                  0.458716   
3                   0.206385         0.000000                  0.368438   
4                   0.284742         0.025000                  0.355845   

   blue_square_shape_margin  
0                 -0.006287  
1                 -0.162716  
2                 -0.163660  
3                 -0.286963  
4                 -0.134209  

Mean:
seed                     

In [91]:
from collections import Counter

for seed in seeds:

    model, _ = train_one_seed(seed)

    set_seed(12345)

    held = evaluate_held_out(
        model=model,
        test_combinations=test_combinations,
        all_combinations=all_combinations,
        tokenizer=tokenizer,
        device=device,
        samples_per_combo=20,
    )

    print(f"\n===== SEED {seed} =====")

    for pair in [
        ("red", "triangle"),
        ("blue", "square"),
    ]:

        subset = [
            r for r in held["results"]
            if r["combo"][:2] == pair
        ]

        counts = Counter(
            r["predicted_caption"]
            for r in subset
        )

        print("\nPAIR:", pair)

        for caption, count in counts.most_common(8):
            print(count, caption)


===== SEED 0 =====

PAIR: ('red', 'triangle')
20 a small red triangle on the left
20 a small red triangle in the center
20 a small red triangle on the right
20 a large red triangle on the left
20 a large red triangle in the center
20 a large red triangle on the right

PAIR: ('blue', 'square')
19 a large blue square on the left
16 a large blue circle in the center
16 a large blue circle on the right
13 a small blue circle on the right
11 a small blue circle on the left
11 a small blue circle in the center
9 a small blue square on the left
9 a small blue square in the center

===== SEED 1 =====

PAIR: ('red', 'triangle')
20 a small red triangle on the left
20 a small red triangle in the center
20 a large red triangle on the left
20 a large red triangle in the center
20 a large red triangle on the right
16 a small red triangle on the right
4 a small red square on the right

PAIR: ('blue', 'square')
20 a small blue circle on the left
20 a small blue circle in the center
20 a small blue ci